## Giao diện Gợi ý Lộ trình Tối ưu bằng Streamlit

In [ ]:
!pip install streamlit folium streamlit-folium pyspark findspark branca

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.5/530.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 48.2 MB/s eta 0:00:00


### Kết nối Google Drive
Cũng giống như pipeline huấn luyện, bạn cần trỏ vào đúng thư mục `Traffic_Project` trên Google Drive để Web Dashboard có thể nạp được thư mục chứa model (`Models/gbt_model`) đã chạy từ trước.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

project_path = '/content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi'
os.makedirs(project_path, exist_ok=True)
os.chdir(project_path)

print(f"Đã trỏ thư mục về: {os.getcwd()}")
print("Đảm bảo bạn đã huấn luyện mô hình (chạy file pipeline) để có sẵn thư mục Models/rf_model tại đây.")

Mounted at /content/drive
Đã trỏ thư mục về: /content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi
Đảm bảo bạn đã huấn luyện mô hình (chạy file pipeline) để có sẵn thư mục Models/rf_model tại đây.


### Tạo file ứng dụng `app.py` trực tiếp lên Drive

In [ ]:
%%writefile app.py
import os
import sys
import requests
import datetime
import math
import numpy as np
import pandas as pd
import streamlit as st
import folium
from folium.plugins import HeatMap, HeatMapWithTime, DualMap
from streamlit_folium import st_folium
import streamlit.components.v1 as stc

# LƯU Ý: BẮT BUỘC GIỮ NGUYÊN MÃ CODE SETUP MÔI TRƯỜNG SPARK
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
python_path = sys.executable.replace('\\', '/')
os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path
os.environ['PYSPARK_PIN_THREAD'] = 'true'

# Monkey-patch cho HeatMapWithTime để sửa lỗi get_bounds()
def patched_hmwt_get_self_bounds(self):
    from branca.utilities import none_min, none_max
    bounds = [[None, None], [None, None]]
    for step in self.data:
        for point in step:
            bounds = [
                [none_min(bounds[0][0], point[0]), none_min(bounds[0][1], point[1])],
                [none_max(bounds[1][0], point[0]), none_max(bounds[1][1], point[1])],
            ]
    return bounds
HeatMapWithTime._get_self_bounds = patched_hmwt_get_self_bounds

# Cấu hình giao diện Web
st.set_page_config(page_title="Dự Báo Tắc Nghẽn Giao Thông", page_icon="🚦", layout="wide", initial_sidebar_state="expanded")

# ================================================================
# QUẢN LÝ TRẠNG THÁI (SESSION STATE) & THEME SETTINGS
# ================================================================
if 'pickup' not in st.session_state: st.session_state.pickup = (40.7589, -73.9851)
if 'dropoff' not in st.session_state: st.session_state.dropoff = (40.7812, -73.9665)
center_lat = (st.session_state.pickup[0] + st.session_state.dropoff[0]) / 2
center_lon = (st.session_state.pickup[1] + st.session_state.dropoff[1]) / 2
if 'click_step' not in st.session_state: st.session_state.click_step = 0
if 'show_results' not in st.session_state: st.session_state.show_results = False
if 'prediction_history' not in st.session_state: st.session_state.prediction_history = []
if 'theme' not in st.session_state: st.session_state.theme = "Cyberpunk"
if 'active_tab' not in st.session_state: st.session_state.active_tab = "🏠 Trang chủ"

themes_dict = {
    "Cyberpunk": {"bg": "linear-gradient(135deg, #0f0c29, #302b63, #24243e)", "text": "#00ffcc", "card": "rgba(0, 255, 204, 0.05)"},
    "Eco Green": {"bg": "linear-gradient(135deg, #d4fc79, #96e6a1)", "text": "#1b4332", "card": "rgba(255, 255, 255, 0.4)"},
    "Dark Mode": {"bg": "linear-gradient(160deg, #0a0a1a 0%, #111133 40%, #1a1a3e 70%, #0d0d24 100%)", "text": "#ffffff", "card": "rgba(255,255,255,0.05)"},
    "Light Mode": {"bg": "linear-gradient(160deg, #f8f9fa 0%, #e9ecef 100%)", "text": "#333333", "card": "rgba(0,0,0,0.05)"},
    "Ocean": {"bg": "linear-gradient(160deg, #004e92 0%, #000428 100%)", "text": "#ffffff", "card": "rgba(255,255,255,0.08)"}
}
current_theme = themes_dict[st.session_state.theme]

st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&display=swap');
    .stApp {{ background: {current_theme['bg']} !important; font-family: 'Inter', sans-serif; color: {current_theme['text']} !important; }}
    h1, h2, h3, h4, .stMarkdown p, .stMetricValue {{ color: {current_theme['text']} !important; }}
    .glass-card {{ background: {current_theme['card']}; backdrop-filter: blur(16px); border: 1px solid rgba(255,255,255,0.1); border-radius: 16px; padding: 20px; margin-bottom: 20px; box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.2); }}
    .stButton > button {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; border: none !important; border-radius: 14px !important; font-weight: 700 !important; transition: all 0.3s ease !important; width: 100%; padding: 0.8rem !important; }}
    .stButton > button:hover {{ transform: translateY(-3px) scale(1.02) !important; box-shadow: 0 10px 20px rgba(0,0,0,0.2); }}
</style>
""", unsafe_allow_html=True)

# ================================================================
# TÍCH HỢP OSRM API - ĐÃ ĐƯỢC FIX ĐỂ ÉP RA ĐƯỜNG THAY THẾ
# ================================================================
@st.cache_data(ttl=3600)
def get_route_osrm(start_coords, end_coords, alternatives=True):
    # Ép lấy tối đa 3 lộ trình thay vì dùng chữ "true" chung chung
    alt_param = "3" if alternatives else "false"
    url = f"http://router.project-osrm.org/route/v1/driving/{start_coords[1]},{start_coords[0]};{end_coords[1]},{end_coords[0]}?overview=full&geometries=geojson&alternatives={alt_param}"
    try:
        r = requests.get(url)
        res = r.json()
        if res.get('code') == 'Ok':
            routes = []
            for route_info in res['routes']:
                route = [[p[1], p[0]] for p in route_info['geometry']['coordinates']]
                dist = route_info['distance'] / 1000.0
                routes.append((route, dist))
            return routes
    except Exception:
        pass
    return [([start_coords, end_coords], 0.0)]

# ================================================================
# KHỞI TẠO SPARK VÀ LOAD MODEL
# ================================================================
@st.cache_resource
def load_spark_and_model():
    try: import findspark; findspark.init()
    except ImportError: pass

    from pyspark.sql import SparkSession
    from pyspark.ml.regression import GBTRegressionModel

    spark, loaded_model = None, None
    try:
        spark = SparkSession.builder.appName("Traffic_Web_App").master("local[1]").config("spark.driver.memory", "10g").config("spark.driver.bindAddress", "127.0.0.1").config("spark.driver.host", "127.0.0.1").getOrCreate()
        spark.sparkContext.setLogLevel("ERROR")
    except Exception: pass

    if spark:
        model_paths = ["/content/drive/MyDrive/Du_bao_tac_duong_giao_thong_do_thi/Models/gbt_model", "Models/gbt_model"]
        for path in model_paths:
            if os.path.exists(path):
                try: loaded_model = GBTRegressionModel.load(path); break
                except Exception: continue
    return spark, loaded_model

spark, model = load_spark_and_model()

if spark is not None and model is not None:
    status_bg, status_color, status_icon, status_text = "rgba(0,204,102,0.2)", "#00cc66", "🟢", "Live: Spark Session Active & GBT Model Online"
elif spark is not None and model is None:
    status_bg, status_color, status_icon, status_text = "rgba(255,75,75,0.2)", "#ff4b4b", "🔴", "Cảnh báo: Spark Active nhưng Model Offline"
else:
    status_bg, status_color, status_icon, status_text = "rgba(255,75,75,0.2)", "#ff4b4b", "🔴", "Offline: Lỗi kết nối Spark Session"

# Header
st.markdown(f"""
<div style="text-align: center; margin-top: 10px;">
    <h1 style="text-shadow: 0 0 15px {current_theme['text']}; font-weight: 800; margin: 10px 0;">Hệ Thống Dự Báo Tắc Nghẽn Đô Thị AI</h1>
    <span style="background: {status_bg}; color:{status_color}; padding: 5px 15px; border-radius: 20px; font-weight:bold; font-size: 0.9em; border: 1px solid {status_color};">{status_icon} {status_text}</span>
</div><br>
""", unsafe_allow_html=True)

menu_options = ["🏠 Trang chủ", "📖 Tài liệu API OSRM", "📍 Chọn địa điểm (Khu 1)", "🧠 Phân tích & Dự đoán", "🕒 Lịch sử"]
st.session_state.active_tab = st.radio("Điều hướng:", menu_options, horizontal=True, label_visibility="collapsed", index=menu_options.index(st.session_state.active_tab))

# Sidebar
with st.sidebar:
    st.header("⚙️ Cài đặt & Tác vụ")
    selected_theme = st.selectbox("🎨 Settings Theme", list(themes_dict.keys()), index=list(themes_dict.keys()).index(st.session_state.theme))
    if selected_theme != st.session_state.theme:
        st.session_state.theme = selected_theme; st.rerun()
    st.markdown("---")
    st.subheader("🕐 Thông số chuyến đi")
    month = st.selectbox("Tháng", range(1, 13), index=4)
    day_of_week = st.selectbox("Ngày trong tuần (1=CN, 2=T2...)", range(1, 8), index=1)
    hour = st.slider("Giờ khởi hành", 0, 23, 17)
    passenger_count = st.number_input("🧑‍🤝‍🧑 Số lượng hành khách", min_value=1, max_value=6, value=1)
    st.markdown("---")
    st.subheader("📍 Nhập tọa độ thủ công")

    def update_coords():
        st.session_state.pickup = (st.session_state.lat_a, st.session_state.lon_a)
        st.session_state.dropoff = (st.session_state.lat_b, st.session_state.lon_b)

    c1, c2 = st.columns(2)
    with c1: st.number_input("Vĩ độ (A)", value=st.session_state.pickup[0], format="%.4f", step=0.001, key="lat_a", on_change=update_coords)
    with c2: st.number_input("Kinh độ (A)", value=st.session_state.pickup[1], format="%.4f", step=0.001, key="lon_a", on_change=update_coords)
    c3, c4 = st.columns(2)
    with c3: st.number_input("Vĩ độ (B)", value=st.session_state.dropoff[0], format="%.4f", step=0.001, key="lat_b", on_change=update_coords)
    with c4: st.number_input("Kinh độ (B)", value=st.session_state.dropoff[1], format="%.4f", step=0.001, key="lon_b", on_change=update_coords)
    st.markdown("<br>", unsafe_allow_html=True)

    if st.button("🚀 Dự báo & gợi ý lộ trình"):
        st.session_state.show_results = True
        st.session_state.force_save = True
        st.session_state.active_tab = "🧠 Phân tích & Dự đoán"
        st.rerun()

center_lat = (st.session_state.pickup[0] + st.session_state.dropoff[0]) / 2
center_lon = (st.session_state.pickup[1] + st.session_state.dropoff[1]) / 2

# NỘI DUNG TRANG
if st.session_state.active_tab == "🏠 Trang chủ":
    st.markdown('<div class="glass-card"><h2>🚩 Giới thiệu chung</h2><p>Hệ thống AI phân tích và dự đoán giao thông...</p></div>', unsafe_allow_html=True)

elif st.session_state.active_tab == "📖 Tài liệu API OSRM":
    st.markdown('<div class="glass-card"><h2>Tiện ÍCH API OSRM</h2><p>Tìm đường đi uốn lượn thực tế và tính toán đường nhánh.</p></div>', unsafe_allow_html=True)

elif st.session_state.active_tab == "📍 Chọn địa điểm (Khu 1)":
    st.markdown('<div class="glass-card">', unsafe_allow_html=True)
    st.subheader("1️⃣ Khu vực 1: Bản đồ Tọa độ (Click để chọn Điểm A & B)")
    m_input = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}', attr='Google')
    folium.Marker(st.session_state.pickup, popup="📍 Điểm đón (A)", icon=folium.Icon(color="green", icon="play")).add_to(m_input)
    folium.Marker(st.session_state.dropoff, popup="🏁 Điểm trả (B)", icon=folium.Icon(color="red", icon="stop")).add_to(m_input)
    map_data = st_folium(m_input, height=500, use_container_width=True, returned_objects=["last_clicked"], key="input_map")

    if map_data and map_data.get("last_clicked"):
        clicked_coord = (map_data["last_clicked"]["lat"], map_data["last_clicked"]["lng"])
        if st.session_state.get('last_click') != clicked_coord:
            st.session_state.last_click = clicked_coord
            if st.session_state.click_step == 0:
                st.session_state.pickup = clicked_coord; st.session_state.click_step = 1
            else:
                st.session_state.dropoff = clicked_coord; st.session_state.click_step = 0
            st.session_state.show_results = False
            st.rerun()
    st.markdown('</div>', unsafe_allow_html=True)

elif st.session_state.active_tab == "🧠 Phân tích & Dự đoán":
    if not st.session_state.show_results:
        st.info("👈 Vui lòng cấu hình tọa độ và nhấn nút **'Dự báo & gợi ý lộ trình'** ở Sidebar.")
    else:
        routes_data = get_route_osrm(st.session_state.pickup, st.session_state.dropoff, alternatives=True)
        primary_route, primary_dist = routes_data[0]

        if model is None:
            st.error("⚠️ Lỗi: Không tìm thấy GBT Model tại thư mục dự án.")
        else:
            # Xử lý Feature & Dự đoán
            lon1, lat1, lon2, lat2 = map(math.radians, [st.session_state.pickup[1], st.session_state.pickup[0], st.session_state.dropoff[1], st.session_state.dropoff[0]])
            haversine_dist = 2 * 6371 * math.asin(math.sqrt(math.sin((lat2-lat1)/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin((lon2-lon1)/2)**2))
            mean_lat = math.radians(center_lat)
            manhattan_dist = abs(st.session_state.pickup[0] - st.session_state.dropoff[0])*111.045 + abs(st.session_state.pickup[1] - st.session_state.dropoff[1])*111.045*math.cos(mean_lat)

            input_data = {
                "passenger_count": float(passenger_count),
                "pickup_longitude": float(st.session_state.pickup[1]), "pickup_latitude": float(st.session_state.pickup[0]),
                "dropoff_longitude": float(st.session_state.dropoff[1]), "dropoff_latitude": float(st.session_state.dropoff[0]),
                "distance_km": float(haversine_dist), "manhattan_distance_km": float(manhattan_dist),
                "day_of_week": float(day_of_week), "month": float(month),
                "hour_sin": float(math.sin(2 * math.pi * hour / 24)), "hour_cos": float(math.cos(2 * math.pi * hour / 24)),
                "vendor_id_indexed": 0.0
            }

            from pyspark.ml.feature import VectorAssembler
            req_df = spark.createDataFrame(pd.DataFrame([input_data]))
            assembler = VectorAssembler(inputCols=list(input_data.keys()), outputCol="features")
            pred_log = model.transform(assembler.transform(req_df)).select("prediction").head()[0]
            pred_result = math.exp(pred_log) - 1 if pred_log > 0 else 0
            minutes, seconds = int(pred_result // 60), int(pred_result % 60)
            actual_dist = primary_dist if primary_dist > 0 else haversine_dist
            speed_kmh = actual_dist / (pred_result / 3600) if pred_result > 0 else 0
            prob_congestion = round(max(0, min(100, (25 - speed_kmh) / 20 * 100)), 1)

            if speed_kmh > 30: route_color, status = "#00cc66", "Thông Suốt"
            elif speed_kmh > 15: route_color, status = "#ffa600", "Ùn Ứ / Chậm"
            else: route_color, status = "#ff4b4b", "Tắc Nghẽn"

            # --- KHU VỰC 3: KPI ---
            st.markdown('<div class="glass-card">', unsafe_allow_html=True)
            st.subheader("3️⃣ Khu vực 3: Kết Quả Dự Báo KPI")
            m1, m2, m3, m4 = st.columns(4)
            m1.metric("📏 Distance (OSRM)", f"{actual_dist:.2f} km")
            m2.metric("⏱️ ETA (Đã tính độ trễ)", f"{minutes}p {seconds}s")
            m3.metric("🚗 Avg Speed", f"{speed_kmh:.1f} km/h")
            m4.metric("🚥 Congestion Index", f"{prob_congestion}% - {status}")
            st.markdown('</div>', unsafe_allow_html=True)

            # --- KHU VỰC 4: BẢN ĐỒ KẾT QUẢ & ĐƯỜNG GỢI Ý (ĐÃ SỬA LỖI UI) ---
            st.markdown('<div class="glass-card">', unsafe_allow_html=True)
            st.subheader("4️⃣ Khu vực 4: Bản Đồ Giao Thông Hybrid (Google Maps)")
            show_alt = False

            # Logic mới: Nếu có tắc nghẽn, kiểm tra xem OSRM có trả về đường nhánh không
            if route_color != "#00cc66":
                if len(routes_data) > 1:
                    st.warning(f"⚠️ Phát hiện đoạn đường đang bị {status}. OSRM đã tính toán một đường nhánh thay thế!")
                    show_alt = st.toggle("🌟 Kích hoạt Gợi ý lộ trình mới (Đường Xanh lá)", value=True)
                else:
                    st.info(f"⚠️ Phát hiện đoạn đường đang bị {status}. Tuy nhiên, API OSRM hiện không tìm thấy lộ trình vòng/nhánh nào khả thi hơn đường chính.")

            m_result = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}', attr='Google')
            folium.Marker(st.session_state.pickup, popup="📍 Điểm đón (A)", icon=folium.Icon(color="green", icon="play")).add_to(m_result)
            folium.Marker(st.session_state.dropoff, popup="🏁 Điểm trả (B)", icon=folium.Icon(color="red", icon="stop")).add_to(m_result)

            folium.PolyLine(locations=primary_route, color=route_color, weight=7, opacity=0.9, tooltip=f"Lộ trình chính: {status}").add_to(m_result)

            if show_alt and len(routes_data) > 1:
                folium.PolyLine(locations=routes_data[1][0], color="#00cc66", weight=6, dash_array='10', opacity=0.9, tooltip="Lộ trình Gợi ý (Thông Suốt)").add_to(m_result)

            st_folium(m_result, height=450, use_container_width=True, key="result_map")
            st.markdown('</div>', unsafe_allow_html=True)

            # --- KHU VỰC 5: LỊCH SỬ ---
            if st.session_state.get('force_save', False):
                st.session_state.prediction_history.append({
                    "Thời gian": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "Quãng đường": f"{actual_dist:.2f} km",
                    "ETA": f"{minutes}p {seconds}s",
                    "Tốc độ": f"{speed_kmh:.1f} km/h",
                    "Ùn tắc": f"{prob_congestion}%",
                    "Trạng thái": status
                })
                st.session_state.force_save = False
            st.markdown('<div class="glass-card">', unsafe_allow_html=True)
            st.subheader("5️⃣ Khu vực 5: Lịch Sử Lần Dự Đoán Vừa Rồi")
            st.dataframe(pd.DataFrame(st.session_state.prediction_history), use_container_width=True)
            st.markdown('</div>', unsafe_allow_html=True)

elif st.session_state.active_tab == "🕒 Lịch sử":
    st.markdown('<div class="glass-card">', unsafe_allow_html=True)
    st.header("🕒 Lịch sử Tra cứu")
    if st.session_state.prediction_history:
        history_df = pd.DataFrame(st.session_state.prediction_history)
        st.dataframe(history_df, use_container_width=True)
    else:
        st.info("Chưa có chuyến đi nào được dự đoán.")
    st.markdown('</div>', unsafe_allow_html=True)

# Footer
st.markdown(f"""<hr style="border-color: rgba(255,255,255,0.1); margin-top: 50px;"><div style="text-align: center; opacity: 0.8; padding: 20px; font-size: 0.9em;"><p><b>© 2026 Bản quyền:</b> Ngô Hữu Phong</p></div>""", unsafe_allow_html=True)

Overwriting app.py


### Chạy Streamlit App và lấy Public URL (Dùng Cloudflare)
Bạn chạy cell dưới đây. Xin chờ khoảng 15-20s để hệ thống boot. Sau đó click vào link đuôi `.trycloudflare.com`.

In [ ]:
# 0. Dọn dẹp tiến trình cũ triệt để
!pkill -f streamlit
!pkill -f cloudflared
!pkill -f localtunnel

import time
import os
time.sleep(2) # Chờ 2 giây để tiến trình cũ tắt hẳn

# ĐẢM BẢO CHUYỂN ĐÚNG THƯ MỤC LÀM VIỆC CÓ CHỨA FILE app.py
# Hãy sửa project_path bên dưới nếu thư mục của bạn trên Drive giữ tên khác
project_path = '/content/drive/MyDrive/Du_doan_tac_nghen_giao_thong_do_thi'
if os.path.exists(project_path):
    os.chdir(project_path)

if not os.path.exists('app.py'):
    print("❌ LỖI NGHIÊM TRỌNG: Không tìm thấy file 'app.py' ở thư mục hiện tại.")
    print(f"Đang đứng tại ổ: {os.getcwd()}")
    print("Giải pháp: Vui lòng dùng lệnh %%writefile app.py (như ở cell trên) hoặc chuyển đúng project_path!")
else:
    # 1. Chạy ngầm Streamlit (đổi address về 0.0.0.0)
    !nohup streamlit run app.py \
      --server.port 8501 \
      --server.address 0.0.0.0 \
      --server.headless true \
      --server.enableCORS false \
      --server.enableXsrfProtection false > logs.txt 2>&1 &

    # 2. Tải trình kết nối Cloudflare về gốc /content để tránh rác Drive
    if not os.path.exists('/content/cloudflared'):
        !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
        !chmod +x /content/cloudflared

    # 3. Kích hoạt đường hầm ra internet
    !nohup /content/cloudflared tunnel --url http://localhost:8501 > /content/tunnel_logs.txt 2>&1 &

    # === QUAN TRỌNG: Tăng thời gian chờ lên để Spark kịp Boot ===
    print("⏳ Đang khởi động Streamlit và Apache Spark engine (mất khoảng 15-20 giây)...")
    time.sleep(18)

    # In ra log của Streamlit để kiểm tra xem nó có bị lỗi (Crash) ngầm hay không
    print("\n--- [LOGS CỦA STREAMLIT] (Đọc log này để biết App có lỗi thiếu thư viện/model không) ---")
    !tail -n 15 logs.txt
    print("--------------------------------------------------------------------------------------\n")

    # In đường link
    print("✅ DASHBOARD CỦA BẠN ĐÃ SẴN SÀNG!")
    print("⚠️ BẮT BUỘC PHẢI CLICK VÀO LINK URL MỚI NHẤT BÊN DƯỚI:")
    !grep -o 'https://[^[:space:]]*\.trycloudflare\.com' /content/tunnel_logs.txt

⏳ Đang khởi động Streamlit và Apache Spark engine (mất khoảng 15-20 giây)...

--- [LOGS CỦA STREAMLIT] (Đọc log này để biết App có lỗi thiếu thư viện/model không) ---


2026-05-14 11:27:42.658 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.190.181.143:8501

--------------------------------------------------------------------------------------

✅ DASHBOARD CỦA BẠN ĐÃ SẴN SÀNG!
⚠️ BẮT BUỘC PHẢI CLICK VÀO LINK URL MỚI NHẤT BÊN DƯỚI:
https://replacing-duo-peninsula-charming.trycloudflare.com
